In [2]:
from pipelines.readers.b3_indices_segmentos_setoriais import ReaderSQLB3IndicesSegmentosSetoriais
from pipelines.readers.cvm_balanco_patrimonial import ReaderCSVCVMBalancoPatrimonial

from yfinance import download
import plotly.express as px
import numpy as np
import pandas as pd
from datetime import date

In [3]:
reader_b3_segmentos_setoriais = ReaderSQLB3IndicesSegmentosSetoriais()
df_b3_segmentos_setoriais = reader_b3_segmentos_setoriais.read(indice="IBEP")

print(df_b3_segmentos_setoriais.shape)
df_b3_segmentos_setoriais.tail(3)

(71, 7)


,Indice,Data Referencia,Código,Ação,Tipo,Qtde. Teórica,Part. (%)
68,IBEP,2026-07-06,VIVT3,TELEF BRASIL,ON EJ,707125712,1.166
69,IBEP,2026-07-06,WEGE3,WEG,ON NM,1485954732,3.278
70,IBEP,2026-07-06,YDUQ3,YDUQS PART,ON NM,261365845,0.112


In [4]:
tickers = list(df_b3_segmentos_setoriais["Código"]) + ["^BVSP"]

In [8]:
reader_csv_balanco_patrimonial = ReaderCSVCVMBalancoPatrimonial()

data = {}

benchmark = download("^BVSP", period="10y", interval="1d", progress=False, auto_adjust=False).droplevel(1, axis=1)[["Adj Close"]].pct_change()

for codigo in tickers:
    
    # Balanco Patrimonial
    try:
        ativo_total = reader_csv_balanco_patrimonial.read(ticker=codigo, tipo_arquivo="BPA_ind", cd_conta="1").astype(int).pct_change().iloc[:, 1].iloc[-1]
    except Exception:
        ativo_total = np.nan
    try:
        resultado_bruto = reader_csv_balanco_patrimonial.read(ticker=codigo, tipo_arquivo="DRE_ind", cd_conta="3.03").astype(int).pct_change().iloc[:, 1].iloc[-1]
    except Exception:
        resultado_bruto = np.nan    
    try:
        receita_de_vendas = reader_csv_balanco_patrimonial.read(ticker=codigo, tipo_arquivo="DRE_ind", cd_conta="3.01").astype(int).pct_change().iloc[:, 1].iloc[-1]
    except Exception:
        receita_de_vendas = np.nan

    try:
        serie_precos = download(codigo + ".SA" if codigo != "^BVSP" else codigo, period="10y", interval="1d", progress=False, auto_adjust=False).droplevel(1, axis=1)
    except Exception:
        
        drawdown_atual = np.nan
        residuo_reg_lin = np.nan

        continue
    
    # Drawdown
    preco_maximo = serie_precos['Adj Close'].max()
    preco_atual = serie_precos['Adj Close'].iloc[-1]
    drawdown_atual = (preco_atual - preco_maximo) / preco_maximo * 100
    
    # Regressão linear
    y = serie_precos["Adj Close"].to_numpy()
    x = np.arange(len(y))

    coef = np.polyfit(x, y, 1)      
    tendencia = np.polyval(coef, x)

    residuo_reg_lin = ((serie_precos["Adj Close"] - tendencia) / tendencia) * 100
    residuo_reg_lin = residuo_reg_lin.iloc[-1]
    
    # Beta
    beta = serie_precos["Adj Close"].pct_change().cov(benchmark["Adj Close"]) / benchmark["Adj Close"].var()
    
    # Retorno agrupado por mês
    serie_precos['ret'] = serie_precos["Adj Close"].pct_change()
    serie_precos['mes'] = serie_precos.index.month
    serie_precos['ano'] = serie_precos.index.year

    tabela = (
        serie_precos.groupby(['ano', 'mes'])[['ret']]
        .sum()
        .reset_index()
        .pivot(index='ano', columns='mes', values='ret')
    )
    
    data[codigo] = {
        "Ativo Total": ativo_total,
        "Resultado Bruto": resultado_bruto,
        "Receita de Vendas": receita_de_vendas,
        "Drawdown Atual": drawdown_atual,
        "Resíduo Regressão Linear": residuo_reg_lin,
        "Beta": beta,
        f"Retorno Agrupado por Mês (Média Mês:{date.today().month})": tabela[date.today().month].mean()
        }
    
    

In [11]:
df_final = (
    df_b3_segmentos_setoriais.set_index("Código")
    .join(pd.DataFrame.from_dict(data, orient="index"))
    .reset_index()
)

benchmark_row = pd.DataFrame([{
    "Código": "^BVSP",
    "Ativo Total": np.nan,
    "Resultado Bruto": np.nan,
    "Receita de Vendas": np.nan,
    "Drawdown Atual": data["^BVSP"]["Drawdown Atual"],
    "Resíduo Regressão Linear": data["^BVSP"]["Resíduo Regressão Linear"],
    "Beta": data["^BVSP"]["Beta"],
    f"Retorno Agrupado por Mês (Média Mês:{date.today().month})": data["^BVSP"][f"Retorno Agrupado por Mês (Média Mês:{date.today().month})"]
}])

df_final = pd.concat([df_final, benchmark_row], ignore_index=True)

pd.set_option("display.max_rows", None)
df_final.sort_values(by="Drawdown Atual", ascending=False)

,Código,Indice,Data Referencia,Ação,Tipo,Qtde. Teórica,Part. (%),Ativo Total,Resultado Bruto,Receita de Vendas,Drawdown Atual,Resíduo Regressão Linear,Beta,Retorno Agrupado por Mês (Média Mês:7)
48,PSSA3,IBEP,2026-07-06,PORTO SEGURO,ON NM,1.788254e+08,0.460,0.056534,NaN,NaN,-2.690218,36.546325,0.539686,0.036751
26,ENEV3,IBEP,2026-07-06,ENEVA,ON NM,1.913352e+09,2.418,0.041386,-0.340257,-0.268461,-3.479525,40.196312,0.747599,0.046497
59,TAEE11,IBEP,2026-07-06,TAESA,UNT N2,2.185682e+08,0.426,0.035111,0.030653,0.020434,-7.238636,8.467384,0.449122,0.042146
0,ABEV3,IBEP,2026-07-06,AMBEV S/A,ON,4.273841e+09,3.304,-0.025933,-0.071207,-0.043674,-7.520953,31.701762,0.650153,0.018904
62,UGPA3,IBEP,2026-07-06,ULTRAPAR,ON NM,1.067870e+09,1.395,0.045903,NaN,NaN,-8.960573,40.189207,1.195638,0.002465
31,GOAU4,IBEP,2026-07-06,GERDAU MET,PN N1,8.308378e+08,0.374,-0.011312,0.288102,-0.209715,-8.987699,8.287816,1.190810,0.086045
36,ITSA4,IBEP,2026-07-06,ITAUSA,PN N1,5.970738e+09,3.853,0.028099,NaN,NaN,-9.135634,39.756540,0.968089,0.036002
18,CPLE3,IBEP,2026-07-06,COPEL,ON NM,2.969407e+09,2.122,0.066665,-0.061209,-0.055854,-10.463758,36.391668,0.837723,0.034592
66,VBBR3,IBEP,2026-07-06,VIBRA,ON NM,1.192005e+09,1.718,0.043912,0.248137,-0.049013,-10.509477,42.134707,1.171826,0.036983
11,BRAP4,IBEP,2026-07-06,BRADESPAR,PN N1,2.509147e+08,0.267,-0.016141,NaN,NaN,-12.004706,6.922039,0.923714,0.047761
